# Employee Attrition Analysis

A portfolio project analyzing employee turnover, satisfaction, overtime, compensation, and workforce patterns.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# --- Visual style: one consistent theme, but a distinct color role per chart type,
# so ranked bars, category bars, trend lines, and comparisons don't all look the same ---
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titlepad'] = 12
plt.rcParams['figure.facecolor'] = 'white'

CATEGORICAL = sns.color_palette("Set2")                  # distinct hues: unordered categories
SEQUENTIAL  = sns.color_palette("crest", as_cmap=True)    # gradient: ranked / sorted bars
ACCENT_A    = "#2E6F95"   # primary trend line
ACCENT_B    = "#E07A5F"   # secondary trend line
GOOD_BAD    = ["#588157", "#BC4749"]   # semantic pair: positive vs negative outcome

def add_bar_labels(ax, fmt="{:,.0f}"):
    for p in ax.patches:
        h = p.get_height()
        if pd.notna(h):
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                        ha='center', va='bottom', fontsize=9, color='#333333',
                        xytext=(0, 3), textcoords='offset points')

## 2. Load Dataset

In [2]:
df = pd.read_csv("employee_attrition_data.csv")
df.head()

,Employee_ID,Department,Job_Level,Age,Years_At_Company,Monthly_Salary,Overtime,Job_Satisfaction,Training_Hours,Remote_Work,Attrition
0,200001,Finance,Associate,59,12,8031.99,No,2.0,1,No,0
1,200002,Operations,Analyst,45,4,8350.91,Yes,4.0,4,No,0
2,200003,HR,Specialist,46,6,3965.67,No,5.0,8,Yes,0
3,200004,IT,Manager,53,3,5818.06,No,3.0,2,No,0
4,200005,Sales,Associate,38,15,7755.01,No,4.0,5,No,0


## 3. Data Understanding

In [3]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

Shape: (7100, 11)

Missing values:
Employee_ID          0
Department          86
Job_Level            0
Age                  0
Years_At_Company     0
Monthly_Salary      73
Overtime             0
Job_Satisfaction    56
Training_Hours       0
Remote_Work          0
Attrition            0
dtype: int64

Duplicate rows: 100

Data types:
Employee_ID           int64
Department           object
Job_Level            object
Age                   int64
Years_At_Company      int64
Monthly_Salary      float64
Overtime             object
Job_Satisfaction    float64
Training_Hours        int64
Remote_Work          object
Attrition             int64
dtype: object


In [4]:
df.describe(include='all')

,Employee_ID,Department,Job_Level,Age,Years_At_Company,Monthly_Salary,Overtime,Job_Satisfaction,Training_Hours,Remote_Work,Attrition
count,7100.000000,7014,7100,7100.000000,7100.000000,7027.000000,7100,7044.000000,7100.000000,7100,7100.000000
unique,NaN,6,5,NaN,NaN,NaN,2,NaN,NaN,2,NaN
top,NaN,Operations,Analyst,NaN,NaN,NaN,No,NaN,NaN,No,NaN
freq,NaN,1502,1510,NaN,NaN,NaN,5115,NaN,NaN,4243,NaN
mean,203451.908451,NaN,NaN,40.630986,7.512113,7058.544871,NaN,3.002697,4.018028,NaN,0.257465
std,2047.364974,NaN,NaN,11.766661,4.615944,2420.657346,NaN,1.423069,2.592253,NaN,0.437268
min,200001.000000,NaN,NaN,-7.000000,0.000000,2500.000000,NaN,1.000000,0.000000,NaN,0.000000
25%,201675.750000,NaN,NaN,31.000000,3.000000,5347.045000,NaN,2.000000,2.000000,NaN,0.000000
50%,203450.500000,NaN,NaN,41.000000,8.000000,7048.240000,NaN,3.000000,4.000000,NaN,0.000000
75%,205225.250000,NaN,NaN,51.000000,12.000000,8683.960000,NaN,4.000000,6.000000,NaN,1.000000


## 4. Data Cleaning

In [5]:
df = df.drop_duplicates().reset_index(drop=True)

for col in ['Department','Job_Level','Overtime','Remote_Work']:
    df[col] = df[col].astype('string').str.strip().str.title()

df['Department'] = df['Department'].fillna('Unknown')

df.loc[(df['Age'] < 18) | (df['Age'] > 100), 'Age'] = np.nan
df['Age'] = df['Age'].fillna(df['Age'].median())

df.loc[df['Monthly_Salary'] <= 0, 'Monthly_Salary'] = np.nan
df['Monthly_Salary'] = df['Monthly_Salary'].fillna(df['Monthly_Salary'].median())

df['Job_Satisfaction'] = df['Job_Satisfaction'].clip(1,5)
df['Job_Satisfaction'] = df['Job_Satisfaction'].fillna(df['Job_Satisfaction'].median())

print(df.isnull().sum())

Employee_ID         0
Department          0
Job_Level           0
Age                 0
Years_At_Company    0
Monthly_Salary      0
Overtime            0
Job_Satisfaction    0
Training_Hours      0
Remote_Work         0
Attrition           0
dtype: int64


## 5. Feature Engineering

In [6]:
df['Age_Group'] = pd.cut(
    df['Age'], [17,25,35,45,55,100],
    labels=['18-25','26-35','36-45','46-55','56+']
)

df['Tenure_Group'] = pd.cut(
    df['Years_At_Company'], [-1,1,3,7,15,100],
    labels=['0-1','2-3','4-7','8-15','16+']
)

df['Salary_Band'] = pd.qcut(
    df['Monthly_Salary'], 4,
    labels=['Low','Mid-Low','Mid-High','High']
)

df.head()

,Employee_ID,Department,Job_Level,Age,Years_At_Company,Monthly_Salary,Overtime,Job_Satisfaction,Training_Hours,Remote_Work,Attrition,Age_Group,Tenure_Group,Salary_Band
0,200001,Finance,Associate,59.0,12,8031.99,No,2.0,1,No,0,56+,8-15,Mid-High
1,200002,Operations,Analyst,45.0,4,8350.91,Yes,4.0,4,No,0,36-45,4-7,Mid-High
2,200003,Hr,Specialist,46.0,6,3965.67,No,5.0,8,Yes,0,46-55,4-7,Low
3,200004,It,Manager,53.0,3,5818.06,No,3.0,2,No,0,46-55,2-3,Mid-Low
4,200005,Sales,Associate,38.0,15,7755.01,No,4.0,5,No,0,36-45,8-15,Mid-High


## 6. Exploratory Data Analysis

In [ ]:
attrition_rate = df['Attrition'].mean() * 100
print(f"Overall attrition rate: {attrition_rate:.2f}%")

counts = df['Attrition'].value_counts().sort_index()
plt.figure(figsize=(6, 4.5))
ax = plt.gca()
ax.bar(['Stayed', 'Left'], counts.values, color=GOOD_BAD, edgecolor='white', linewidth=0.6, width=0.55)
ax.set_title('Employee Attrition Split')
ax.set_ylabel('Employees')
add_bar_labels(ax, "{:,.0f}")
plt.tight_layout()
plt.show()

In [ ]:
dept_attrition = df.groupby('Department')['Attrition'].mean().mul(100).sort_values(ascending=False)

colors = SEQUENTIAL(np.linspace(0.25, 0.9, len(dept_attrition)))
plt.figure(figsize=(8, 4.5))
ax = plt.gca()
ax.bar(dept_attrition.index, dept_attrition.values, color=colors, edgecolor='white', linewidth=0.6)
ax.set_title('Attrition Rate by Department')
ax.set_ylabel('Attrition Rate (%)')
plt.xticks(rotation=20)
add_bar_labels(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

In [ ]:
overtime_attrition = df.groupby('Overtime')['Attrition'].mean().mul(100)

plt.figure(figsize=(6, 4.5))
ax = plt.gca()
ax.bar(overtime_attrition.index.astype(str), overtime_attrition.values,
       color=CATEGORICAL[:len(overtime_attrition)], edgecolor='white', linewidth=0.6, width=0.55)
ax.set_title('Attrition Rate by Overtime')
ax.set_ylabel('Attrition Rate (%)')
add_bar_labels(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

In [ ]:
sat_attrition = df.groupby('Job_Satisfaction')['Attrition'].mean().mul(100)

plt.figure(figsize=(7, 4.5))
ax = plt.gca()
ax.plot(sat_attrition.index, sat_attrition.values, marker='o', color=ACCENT_A,
        linewidth=2.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5)
ax.fill_between(sat_attrition.index, sat_attrition.values, color=ACCENT_A, alpha=0.12)
ax.set_title('Attrition Rate by Job Satisfaction')
ax.set_xlabel('Satisfaction Score')
ax.set_ylabel('Attrition Rate (%)')
plt.xticks(range(1, 6))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
ax = sns.boxplot(data=df, x='Attrition', y='Monthly_Salary', hue='Attrition',
                  palette=GOOD_BAD, legend=False)
ax.set_title('Monthly Salary by Attrition Status')
ax.set_xlabel('Attrition (0 = Stayed, 1 = Left)')
ax.set_xticklabels(['Stayed', 'Left'])
plt.tight_layout()
plt.show()

## 7. Statistical Analysis

In [12]:
stayed = df.loc[df['Attrition']==0, 'Monthly_Salary']
left = df.loc[df['Attrition']==1, 'Monthly_Salary']

t_stat, p_value = stats.ttest_ind(stayed, left, equal_var=False)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.5f}")
print("Statistically significant." if p_value < 0.05 else "Not statistically significant.")

T-statistic: 4.359
P-value: 0.00001
Statistically significant.


## 8. KPI Summary

In [13]:
pd.Series({
    'Employees': df['Employee_ID'].nunique(),
    'Attrition Rate (%)': df['Attrition'].mean()*100,
    'Average Salary': df['Monthly_Salary'].mean(),
    'Average Tenure': df['Years_At_Company'].mean(),
    'Average Satisfaction': df['Job_Satisfaction'].mean()
}).round(2)

Employees               7000.00
Attrition Rate (%)        25.76
Average Salary          7056.98
Average Tenure             7.52
Average Satisfaction       3.00
dtype: float64

## 9. Business Insights

- Compare attrition across departments.
- Examine overtime and job satisfaction as possible turnover indicators.
- Compare salary distributions for employees who stayed and left.
- Use the strongest patterns to prioritize retention actions.

## 10. Executive Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Employee Attrition Analytics Dashboard', fontsize=18, fontweight='bold')

axes[0, 0].bar(dept_attrition.index, dept_attrition.values,
               color=SEQUENTIAL(np.linspace(0.25, 0.9, len(dept_attrition))), edgecolor='white', linewidth=0.6)
axes[0, 0].set_title('Attrition by Department')
axes[0, 0].tick_params(axis='x', rotation=20)

axes[0, 1].bar(overtime_attrition.index.astype(str), overtime_attrition.values,
               color=CATEGORICAL[:len(overtime_attrition)], edgecolor='white', linewidth=0.6, width=0.55)
axes[0, 1].set_title('Attrition by Overtime')

axes[1, 0].plot(sat_attrition.index, sat_attrition.values, marker='o', color=ACCENT_A, linewidth=2.2,
                markersize=6, markerfacecolor='white', markeredgewidth=1.5)
axes[1, 0].fill_between(sat_attrition.index, sat_attrition.values, color=ACCENT_A, alpha=0.12)
axes[1, 0].set_title('Attrition vs Satisfaction')

axes[1, 1].bar(
    ['Stayed', 'Left'],
    [df.loc[df['Attrition'] == 0, 'Monthly_Salary'].mean(),
     df.loc[df['Attrition'] == 1, 'Monthly_Salary'].mean()],
    color=GOOD_BAD, edgecolor='white', linewidth=0.6, width=0.55
)
axes[1, 1].set_title('Average Salary by Attrition')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()